In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
BASE = "/content/drive/MyDrive/phishing-project"
os.makedirs(f"{BASE}/data", exist_ok=True)
os.makedirs(f"{BASE}/models", exist_ok=True)
os.makedirs(f"{BASE}/results", exist_ok=True)
print("Ready:", BASE)

Ready: /content/drive/MyDrive/phishing-project


In [4]:
import glob
print(glob.glob(f"{BASE}/data/*.csv"))

['/content/drive/MyDrive/phishing-project/data/Phishing_Email.csv']


In [8]:
import pandas as pd

BASE = "/content/drive/MyDrive/phishing-project"
train = pd.read_csv(f"{BASE}/data/train.csv")
val   = pd.read_csv(f"{BASE}/data/val.csv")
test  = pd.read_csv(f"{BASE}/data/test.csv")

print(train.shape, val.shape, test.shape)

(11190, 3) (3730, 3) (3730, 3)


In [9]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

def evaluate(y_true, y_pred, y_score=None, name="model"):
    res = {
        "model": name,
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall":    recall_score(y_true, y_pred),
        "f1":        f1_score(y_true, y_pred),
    }
    if y_score is not None:
        res["roc_auc"] = roc_auc_score(y_true, y_score)
    print(res)
    print(confusion_matrix(y_true, y_pred))
    return res

In [ ]:
# Initialize the results list
results = []

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2),
                        stop_words="english", sublinear_tf=True)

X_train = tfidf.fit_transform(train["Email Text"].astype(str))   # fit on train ONLY
X_val   = tfidf.transform(val["Email Text"].astype(str))
X_test  = tfidf.transform(test["Email Text"].astype(str))

# The results list is initialized in a previous cell (0f3147dd)

# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, train["Email Type"])
results.append(evaluate(val["Email Type"], lr.predict(X_val),
                        lr.predict_proba(X_val)[:, 1], "TFIDF+LogReg (val)"))

# Linear SVM
svm = LinearSVC()
svm.fit(X_train, train["Email Type"])
results.append(evaluate(val["Email Type"], svm.predict(X_val),
                        svm.decision_function(X_val), "TFIDF+SVM (val)"))

KeyError: 'text'

In [11]:
print(train.columns.tolist())
print(train.dtypes)
print(train.iloc[:, -1].unique()[:10])
print(train.isna().sum())

['Unnamed: 0', 'Email Text', 'Email Type']
Unnamed: 0     int64
Email Text    object
Email Type    object
dtype: object
['Phishing Email' 'Safe Email']
Unnamed: 0     0
Email Text    10
Email Type     0
dtype: int64


In [6]:
from sklearn.model_selection import train_test_split

# Load the available Phishing_Email.csv file
df = pd.read_csv(f"{BASE}/data/Phishing_Email.csv")

# Split the data into training, validation, and test sets
train_val, test = train_test_split(df, test_size=0.2, random_state=42)
train, val = train_test_split(train_val, test_size=0.25, random_state=42) # 0.25 * 0.8 = 0.2 of original data

# Save the split datasets
train.to_csv(f"{BASE}/data/train.csv", index=False)
val.to_csv(f"{BASE}/data/val.csv", index=False)
test.to_csv(f"{BASE}/data/test.csv", index=False)

print(f"Original data shape: {df.shape}")
print(f"Train data shape: {train.shape}")
print(f"Validation data shape: {val.shape}")
print(f"Test data shape: {test.shape}")

Original data shape: (18650, 3)
Train data shape: (11190, 3)
Validation data shape: (3730, 3)
Test data shape: (3730, 3)


In [14]:
print(train["Email Type"].unique())
print(train["Email Type"].value_counts(dropna=False))
print(train["Email Text"].isna().sum(), "missing texts")

['Phishing Email' 'Safe Email']
Email Type
Safe Email        6786
Phishing Email    4404
Name: count, dtype: int64
10 missing texts


In [15]:
label_map = {"Safe Email": 0, "Phishing Email": 1}   # must match the printed values exactly

for name in ["train", "val", "test"]:
    d = globals()[name]
    d["Email Type"] = d["Email Type"].map(label_map)
    d.dropna(subset=["Email Text", "Email Type"], inplace=True)
    d["Email Type"] = d["Email Type"].astype(int)
    d.reset_index(drop=True, inplace=True)

print(train["Email Type"].value_counts())
print(len(train), len(val), len(test))

Email Type
0    6786
1    4394
Name: count, dtype: int64
11180 3726 3728


In [16]:
print(train["Email Text"].duplicated().sum(), "duplicates in train")
overlap = set(train["Email Text"]) & set(test["Email Text"])
print(len(overlap), "emails appear in both train and test")

539 duplicates in train
131 emails appear in both train and test


In [17]:
!pip install -q transformers

import torch, numpy as np, pandas as pd, random
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

TEXT_COL, LABEL_COL = "Email Text", "Email Type"
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN, BATCH, EPOCHS, LR, SEED = 256, 16, 3, 2e-5, 42
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)   # must print cuda

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

cuda


In [18]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class EmailDS(Dataset):
    def __init__(self, df):
        self.texts  = df[TEXT_COL].astype(str).tolist()
        self.labels = df[LABEL_COL].astype(int).tolist()
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        enc = tokenizer(self.texts[i], truncation=True, max_length=MAX_LEN,
                        padding="max_length", return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item

train_dl = DataLoader(EmailDS(train), batch_size=BATCH, shuffle=True)
val_dl   = DataLoader(EmailDS(val),   batch_size=32)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [19]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_dl) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
scaler = torch.cuda.amp.GradScaler()

def predict(dl):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for b in dl:
            b = {k: v.to(device) for k, v in b.items()}
            with torch.autocast("cuda", dtype=torch.float16):
                out = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"])
            probs.extend(torch.softmax(out.logits.float(), dim=1)[:, 1].cpu().numpy())
            labels.extend(b["labels"].cpu().numpy())
    return np.array(labels), np.array(probs)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_3032/1904995832.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [20]:
best_f1 = 0
for epoch in range(EPOCHS):
    model.train()
    running = 0
    for step, b in enumerate(train_dl):
        b = {k: v.to(device) for k, v in b.items()}
        optimizer.zero_grad()
        with torch.autocast("cuda", dtype=torch.float16):
            out = model(**b)
        scaler.scale(out.loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running += out.loss.item()
        if step % 100 == 0:
            print(f"epoch {epoch+1} step {step}/{len(train_dl)} loss {running/(step+1):.4f}")

    y, p = predict(val_dl)
    from sklearn.metrics import f1_score
    f1 = f1_score(y, (p > 0.5).astype(int))
    print(f"--- epoch {epoch+1} val F1: {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), f"{BASE}/models/distilbert_best.pt")

epoch 1 step 0/699 loss 0.6894
epoch 1 step 100/699 loss 0.5913
epoch 1 step 200/699 loss 0.3886
epoch 1 step 300/699 loss 0.2996
epoch 1 step 400/699 loss 0.2463
epoch 1 step 500/699 loss 0.2184
epoch 1 step 600/699 loss 0.1994
--- epoch 1 val F1: 0.9685
epoch 2 step 0/699 loss 0.0061
epoch 2 step 100/699 loss 0.0440
epoch 2 step 200/699 loss 0.0549
epoch 2 step 300/699 loss 0.0552
epoch 2 step 400/699 loss 0.0528
epoch 2 step 500/699 loss 0.0496
epoch 2 step 600/699 loss 0.0479
--- epoch 2 val F1: 0.9580
epoch 3 step 0/699 loss 0.0149
epoch 3 step 100/699 loss 0.0247
epoch 3 step 200/699 loss 0.0251
epoch 3 step 300/699 loss 0.0268
epoch 3 step 400/699 loss 0.0281
epoch 3 step 500/699 loss 0.0276
epoch 3 step 600/699 loss 0.0273
--- epoch 3 val F1: 0.9702


In [24]:
model.load_state_dict(torch.load(f"{BASE}/models/distilbert_best.pt"))
y, p = predict(val_dl)
res = evaluate(y, (p > 0.5).astype(int), p, "DistilBERT (val)")
results.append(res)
pd.DataFrame(results).to_csv(f"{BASE}/results/baselines_val.csv", index=False)

{'model': 'DistilBERT (val)', 'accuracy': 0.97638217928073, 'precision': 0.9610999329309189, 'recall': 0.9794941900205059, 'f1': 0.970209884901828, 'roc_auc': np.float64(0.9978511034747517)}
[[2205   58]
 [  30 1433]]


NameError: name 'results' is not defined

In [22]:
import os
print(os.path.exists(f"{BASE}/models/distilbert_best.pt"))
print(os.listdir(f"{BASE}/models"))

True
['distilbert_best.pt']


In [23]:
for name in ["BASE", "model", "predict", "val_dl", "evaluate", "results", "device"]:
    print(name, "OK" if name in globals() else "MISSING")

BASE OK
model OK
predict OK
val_dl OK
evaluate OK
results MISSING
device OK


In [7]:
# Now that the files are created, re-run the original cell to load them

BASE = "/content/drive/MyDrive/phishing-project"
train = pd.read_csv(f"{BASE}/data/train.csv")
val   = pd.read_csv(f"{BASE}/data/val.csv")
test  = pd.read_csv(f"{BASE}/data/test.csv")

print(train.shape, val.shape, test.shape)

(11190, 3) (3730, 3) (3730, 3)


In [25]:
import os
print("BASE:", BASE if "BASE" in globals() else "MISSING")
print("models folder exists:", os.path.exists(f"{BASE}/models"))
print("files:", os.listdir(f"{BASE}/models") if os.path.exists(f"{BASE}/models") else "none")

for name in ["model", "predict", "val_dl", "evaluate", "results", "device"]:
    print(name, "OK" if name in globals() else "MISSING")

BASE: /content/drive/MyDrive/phishing-project
models folder exists: True
files: ['distilbert_best.pt']
model OK
predict OK
val_dl OK
evaluate OK
results MISSING
device OK


In [26]:
results = [] # Initialize the results list, as it was not defined
model.load_state_dict(torch.load(f"{BASE}/models/distilbert_best.pt", map_location=device))
y, p = predict(val_dl)
res = evaluate(y, (p > 0.5).astype(int), p, "DistilBERT (val)")
results.append(res)
pd.DataFrame(results).to_csv(f"{BASE}/results/baselines_val.csv", index=False)

{'model': 'DistilBERT (val)', 'accuracy': 0.97638217928073, 'precision': 0.9610999329309189, 'recall': 0.9794941900205059, 'f1': 0.970209884901828, 'roc_auc': np.float64(0.9978511034747517)}
[[2205   58]
 [  30 1433]]


NameError: name 'results' is not defined

In [27]:
import pandas as pd

results = []          # recreate the list
results.append(res)   # res still holds the DistilBERT scores

pd.DataFrame(results).to_csv(f"{BASE}/results/baselines_val.csv", index=False)
pd.DataFrame(results)

,model,accuracy,precision,recall,f1,roc_auc
0,DistilBERT (val),0.976382,0.9611,0.979494,0.97021,0.997851


In [31]:
import torch.nn as nn
from transformers import AutoModel
from sklearn.metrics import f1_score

class HybridModel(nn.Module):
    def __init__(self, gru_units=128, dropout=0.3, heads=8, cnn_filters=128):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        hid = self.bert.config.hidden_size
        self.attn = nn.MultiheadAttention(hid, heads, dropout=0.1, batch_first=True)
        self.norm = nn.LayerNorm(hid)
        self.conv = nn.Conv1d(hid, cnn_filters, kernel_size=3, padding=1)
        self.bn = nn.BatchNorm1d(cnn_filters)
        self.gru = nn.GRU(cnn_filters, gru_units, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.fc1 = nn.Linear(4 * gru_units, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        h = out.last_hidden_state
        pad = attention_mask == 0
        attn_out = self.attn(h, h, h, key_padding_mask=pad, need_weights=False)
        h = self.norm(h + attn_out[0])
        c = self.conv(h.transpose(1, 2))
        c = torch.relu(self.bn(c))
        c = c.transpose(1, 2)
        g, _ = self.gru(c)
        m = attention_mask.unsqueeze(-1).to(g.dtype)
        mean = (g * m).sum(1) / m.sum(1).clamp(min=1)
        g_masked = g.masked_fill(m == 0, -1e4)
        mx = g_masked.max(1).values
        feats = torch.cat([mean, mx], dim=1)
        feats = self.drop(feats)
        z = torch.relu(self.fc1(feats))
        z = self.drop(z)
        return self.fc2(z).squeeze(-1)

In [32]:
def predict_hybrid(model, dl):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for b in dl:
            b = {k: v.to(device) for k, v in b.items()}
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(b["input_ids"], b["attention_mask"])
            probs.extend(torch.sigmoid(logits.float()).cpu().numpy())
            labels.extend(b["labels"].cpu().numpy())
    return np.array(labels), np.array(probs)

def run_training(cfg, epochs=3, save_path=None):
    model = HybridModel(cfg["gru_units"], cfg["dropout"]).to(device)
    bert_params = list(model.bert.parameters())
    head_params = [p for n, p in model.named_parameters() if not n.startswith("bert.")]
    optimizer = AdamW([{"params": bert_params, "lr": cfg["bert_lr"]},
                       {"params": head_params, "lr": cfg["head_lr"]}], weight_decay=0.01)
    total = len(train_dl) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total), total)
    scaler = torch.cuda.amp.GradScaler()
    loss_fn = nn.BCEWithLogitsLoss()
    best = 0.0

    for epoch in range(epochs):
        model.train()
        running = 0
        for step, b in enumerate(train_dl):
            b = {k: v.to(device) for k, v in b.items()}
            optimizer.zero_grad()
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(b["input_ids"], b["attention_mask"])
            loss = loss_fn(logits.float(), b["labels"].float())
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running += loss.item()
            if step % 100 == 0:
                print(f"epoch {epoch+1} step {step}/{len(train_dl)} loss {running/(step+1):.4f}")

        y, p = predict_hybrid(model, val_dl)
        f1 = f1_score(y, (p > 0.5).astype(int))
        print(f"--- epoch {epoch+1} val F1: {f1:.4f}")
        if f1 > best:
            best = f1
            if save_path:
                torch.save(model.state_dict(), save_path)
    return model, best

In [33]:
import gc
try:
    del model, optimizer, scheduler, scaler     # old DistilBERT objects
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

cfg = {"gru_units": 128, "dropout": 0.3, "bert_lr": 2e-5, "head_lr": 1e-3}
hybrid, best_f1 = run_training(cfg, epochs=3, save_path=f"{BASE}/models/hybrid_best.pt")
print("best val F1:", best_f1)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_3032/2446373937.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


epoch 1 step 0/699 loss 0.6821
epoch 1 step 100/699 loss 0.4218
epoch 1 step 200/699 loss 0.3149
epoch 1 step 300/699 loss 0.2768
epoch 1 step 400/699 loss 0.2525
epoch 1 step 500/699 loss 0.2299
epoch 1 step 600/699 loss 0.2083
--- epoch 1 val F1: 0.9507
epoch 2 step 0/699 loss 0.0164
epoch 2 step 100/699 loss 0.0823
epoch 2 step 200/699 loss 0.0707
epoch 2 step 300/699 loss 0.0747
epoch 2 step 400/699 loss 0.0761
epoch 2 step 500/699 loss 0.0783
epoch 2 step 600/699 loss 0.0777
--- epoch 2 val F1: 0.9687
epoch 3 step 0/699 loss 0.0058
epoch 3 step 100/699 loss 0.0267
epoch 3 step 200/699 loss 0.0411
epoch 3 step 300/699 loss 0.0443
epoch 3 step 400/699 loss 0.0396
epoch 3 step 500/699 loss 0.0399
epoch 3 step 600/699 loss 0.0401
--- epoch 3 val F1: 0.9698
best val F1: 0.9698189134808853


In [34]:
hybrid.load_state_dict(torch.load(f"{BASE}/models/hybrid_best.pt", map_location=device))
y, p = predict_hybrid(hybrid, val_dl)
res = evaluate(y, (p > 0.5).astype(int), p, "Hybrid BERT+Attn+CNN+BiGRU (val)")

if "results" not in globals():
    results = []
results.append(res)
pd.DataFrame(results).to_csv(f"{BASE}/results/baselines_val.csv", index=False)
pd.DataFrame(results)

{'model': 'Hybrid BERT+Attn+CNN+BiGRU (val)', 'accuracy': 0.9758454106280193, 'precision': 0.9519420671494404, 'recall': 0.9883800410116199, 'f1': 0.9698189134808853, 'roc_auc': np.float64(0.9974107224031638)}
[[2190   73]
 [  17 1446]]


,model,accuracy,precision,recall,f1,roc_auc
0,DistilBERT (val),0.976382,0.961100,0.979494,0.970210,0.997851
1,Hybrid BERT+Attn+CNN+BiGRU (val),0.975845,0.951942,0.988380,0.969819,0.997411


In [35]:
!pip install -q optuna

import optuna, gc

tune_df = train.sample(frac=0.3, random_state=42)
tune_dl = DataLoader(EmailDS(tune_df), batch_size=BATCH, shuffle=True)
print(len(tune_df), "emails,", len(tune_dl), "steps per epoch")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 7.8 MB/s eta 0:00:00
3354 emails, 210 steps per epoch


In [36]:
def run_training(cfg, epochs=3, save_path=None, loader=None):
    if loader is None:
        loader = train_dl
    model = HybridModel(cfg["gru_units"], cfg["dropout"]).to(device)
    bert_params = list(model.bert.parameters())
    head_params = []
    for n, p in model.named_parameters():
        if not n.startswith("bert."):
            head_params.append(p)
    groups = [
        {"params": bert_params, "lr": cfg["bert_lr"]},
        {"params": head_params, "lr": cfg["head_lr"]},
    ]
    optimizer = AdamW(groups, weight_decay=0.01)
    total = len(loader) * epochs
    warm = int(0.1 * total)
    scheduler = get_linear_schedule_with_warmup(optimizer, warm, total)
    scaler = torch.cuda.amp.GradScaler()
    loss_fn = nn.BCEWithLogitsLoss()
    best = 0.0

    for epoch in range(epochs):
        model.train()
        for step, b in enumerate(loader):
            b = {k: v.to(device) for k, v in b.items()}
            optimizer.zero_grad()
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(b["input_ids"], b["attention_mask"])
            loss = loss_fn(logits.float(), b["labels"].float())
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            if step % 100 == 0:
                print(f"epoch {epoch+1} step {step}/{len(loader)} loss {loss.item():.4f}")

        y, p = predict_hybrid(model, val_dl)
        f1 = f1_score(y, (p > 0.5).astype(int))
        print(f"--- epoch {epoch+1} val F1: {f1:.4f}")
        if f1 > best:
            best = f1
            if save_path:
                torch.save(model.state_dict(), save_path)
    return model, best

In [37]:
def objective(trial):
    cfg = {
        "gru_units": trial.suggest_categorical("gru_units", [64, 128, 256]),
        "dropout": trial.suggest_float("dropout", 0.1, 0.5),
        "bert_lr": trial.suggest_float("bert_lr", 1e-5, 5e-5, log=True),
        "head_lr": trial.suggest_float("head_lr", 3e-4, 3e-3, log=True),
    }
    model, best = run_training(cfg, epochs=2, loader=tune_dl)
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return best

db = f"sqlite:///{BASE}/results/optuna.db"
study = optuna.create_study(
    study_name="hybrid_tuning",
    direction="maximize",
    storage=db,
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=15)

print("Best F1:", study.best_value)
print("Best params:", study.best_params)
study.trials_dataframe().to_csv(f"{BASE}/results/optuna_trials.csv", index=False)

[I 2026-09-30 20:33:00,234] A new study created in RDB with name: hybrid_tuning


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_3032/1850922436.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


epoch 1 step 0/210 loss 0.7033
epoch 1 step 100/210 loss 0.0334
epoch 1 step 200/210 loss 0.2351
--- epoch 1 val F1: 0.9211
epoch 2 step 0/210 loss 0.0399
epoch 2 step 100/210 loss 0.0132
epoch 2 step 200/210 loss 0.2261
--- epoch 2 val F1: 0.9484


[I 2026-09-30 20:34:51,691] Trial 0 finished with value: 0.948397976391231 and parameters: {'gru_units': 128, 'dropout': 0.3394633936788146, 'bert_lr': 1.2854415975274441e-05, 'head_lr': 0.00042965094867334556}. Best is trial 0 with value: 0.948397976391231.


...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_3032/1850922436.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


epoch 1 step 0/210 loss 0.7166
epoch 1 step 100/210 loss 0.0390
epoch 1 step 200/210 loss 0.2070
--- epoch 1 val F1: 0.9222
epoch 2 step 0/210 loss 0.0740
epoch 2 step 100/210 loss 0.0269
epoch 2 step 200/210 loss 0.0042
--- epoch 2 val F1: 0.9621


[I 2026-09-30 20:53:53,362] Trial 14 finished with value: 0.962059620596206 and parameters: {'gru_units': 64, 'dropout': 0.3057079140073582, 'bert_lr': 4.382663026230385e-05, 'head_lr': 0.0007206645491440931}. Best is trial 8 with value: 0.9625167336010709.


Best F1: 0.9625167336010709
Best params: {'gru_units': 256, 'dropout': 0.4100531293444458, 'bert_lr': 4.536089127921625e-05, 'head_lr': 0.002354770601447807}


In [38]:
best_cfg = {
    "gru_units": study.best_params["gru_units"],
    "dropout": study.best_params["dropout"],
    "bert_lr": study.best_params["bert_lr"],
    "head_lr": study.best_params["head_lr"],
}
print(best_cfg)

tuned, f1 = run_training(
    best_cfg, epochs=3, save_path=f"{BASE}/models/hybrid_tuned.pt"
)
print("tuned val F1:", f1)

{'gru_units': 256, 'dropout': 0.4100531293444458, 'bert_lr': 4.536089127921625e-05, 'head_lr': 0.002354770601447807}


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_3032/1850922436.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


epoch 1 step 0/699 loss 0.6955
epoch 1 step 100/699 loss 0.5614
epoch 1 step 200/699 loss 0.0295
epoch 1 step 300/699 loss 0.1301
epoch 1 step 400/699 loss 0.1447
epoch 1 step 500/699 loss 0.2608
epoch 1 step 600/699 loss 0.0123
--- epoch 1 val F1: 0.9451
epoch 2 step 0/699 loss 0.0009
epoch 2 step 100/699 loss 0.0928
epoch 2 step 200/699 loss 0.0016
epoch 2 step 300/699 loss 0.0012
epoch 2 step 400/699 loss 0.0467
epoch 2 step 500/699 loss 0.0007
epoch 2 step 600/699 loss 0.0009
--- epoch 2 val F1: 0.9652
epoch 3 step 0/699 loss 0.0290
epoch 3 step 100/699 loss 0.0106
epoch 3 step 200/699 loss 0.0002
epoch 3 step 300/699 loss 0.0003
epoch 3 step 400/699 loss 0.0001
epoch 3 step 500/699 loss 0.0575
epoch 3 step 600/699 loss 0.0003
--- epoch 3 val F1: 0.9716
tuned val F1: 0.9716024340770791


In [39]:
tuned.load_state_dict(
    torch.load(f"{BASE}/models/hybrid_tuned.pt", map_location=device)
)
y, p = predict_hybrid(tuned, val_dl)
res = evaluate(y, (p > 0.5).astype(int), p, "Hybrid + Optuna (val)")

if "results" not in globals():
    results = []
results.append(res)
pd.DataFrame(results).to_csv(f"{BASE}/results/baselines_val.csv", index=False)
pd.DataFrame(results)

{'model': 'Hybrid + Optuna (val)', 'accuracy': 0.9774557165861514, 'precision': 0.9612040133779264, 'recall': 0.9822282980177717, 'f1': 0.9716024340770791, 'roc_auc': np.float64(0.9974786824450755)}
[[2205   58]
 [  26 1437]]


,model,accuracy,precision,recall,f1,roc_auc
0,DistilBERT (val),0.976382,0.961100,0.979494,0.970210,0.997851
1,Hybrid BERT+Attn+CNN+BiGRU (val),0.975845,0.951942,0.988380,0.969819,0.997411
2,Hybrid + Optuna (val),0.977456,0.961204,0.982228,0.971602,0.997479


In [40]:
import random, gc, os
import numpy as np
import torch.nn as nn
from transformers import AutoModel

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

class AblModel(nn.Module):
    def __init__(self, gru_units, dropout, use_attn=True,
                 use_cnn=True, use_gru=True, heads=8, filters=128):
        super().__init__()
        self.use_attn = use_attn
        self.use_cnn = use_cnn
        self.use_gru = use_gru
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        hid = self.bert.config.hidden_size
        self.attn = nn.MultiheadAttention(
            hid, heads, dropout=0.1, batch_first=True)
        self.norm = nn.LayerNorm(hid)
        self.conv = nn.Conv1d(hid, filters, kernel_size=3, padding=1)
        self.bn = nn.BatchNorm1d(filters)
        dim = filters if use_cnn else hid
        if use_gru:
            self.gru = nn.GRU(dim, gru_units, batch_first=True,
                              bidirectional=True)
            dim = 2 * gru_units
        self.drop = nn.Dropout(dropout)
        self.fc1 = nn.Linear(2 * dim, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        h = out.last_hidden_state
        if self.use_attn:
            pad = attention_mask == 0
            a = self.attn(h, h, h, key_padding_mask=pad, need_weights=False)
            h = self.norm(h + a[0])
        if self.use_cnn:
            c = self.conv(h.transpose(1, 2))
            h = torch.relu(self.bn(c)).transpose(1, 2)
        if self.use_gru:
            h, _ = self.gru(h)
        m = attention_mask.unsqueeze(-1).to(h.dtype)
        mean = (h * m).sum(1) / m.sum(1).clamp(min=1)
        mx = h.masked_fill(m == 0, -1e4).max(1).values
        feats = self.drop(torch.cat([mean, mx], dim=1))
        z = self.drop(torch.relu(self.fc1(feats)))
        return self.fc2(z).squeeze(-1)

In [41]:
from torch.optim import AdamW
from sklearn.metrics import f1_score

def run_abl(cfg, flags, seed, epochs, save_path):
    set_seed(seed)
    model = AblModel(cfg["gru_units"], cfg["dropout"], **flags).to(device)
    bert_p = list(model.bert.parameters())
    head_p = []
    for n, p in model.named_parameters():
        if not n.startswith("bert."):
            head_p.append(p)
    groups = [
        {"params": bert_p, "lr": cfg["bert_lr"]},
        {"params": head_p, "lr": cfg["head_lr"]},
    ]
    opt = AdamW(groups, weight_decay=0.01)
    total = len(train_dl) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    scaler = torch.cuda.amp.GradScaler()
    loss_fn = nn.BCEWithLogitsLoss()
    best = 0.0
    for epoch in range(epochs):
        model.train()
        for step, b in enumerate(train_dl):
            b = {k: v.to(device) for k, v in b.items()}
            opt.zero_grad()
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(b["input_ids"], b["attention_mask"])
            loss = loss_fn(logits.float(), b["labels"].float())
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            if step % 200 == 0:
                print(f"  epoch {epoch+1} step {step}/{len(train_dl)} loss {loss.item():.4f}")
        y, p = predict_hybrid(model, val_dl)
        f1 = f1_score(y, (p > 0.5).astype(int))
        print(f"  --- epoch {epoch+1} val F1: {f1:.4f}")
        if f1 > best:
            best = f1
            torch.save(model.state_dict(), save_path)
    return model

In [1]:
try:
    cfg = best_cfg
except NameError:
    import optuna
    db = f"sqlite:///{BASE}/results/optuna.db"
    st = optuna.load_study(study_name="hybrid_tuning", storage=db)
    bp = st.best_params
    cfg = {
        "gru_units": bp["gru_units"],
        "dropout": bp["dropout"],
        "bert_lr": bp["bert_lr"],
        "head_lr": bp["head_lr"],
    }
print("Using:", cfg)

experiments = [
    ("full_seed42", {}, 42),
    ("full_seed43", {}, 43),
    ("full_seed44", {}, 44),
    ("no_attention", {"use_attn": False}, 42),
    ("no_cnn", {"use_cnn": False}, 42),
    ("no_bigru", {"use_gru": False}, 42),
]

out_csv = f"{BASE}/results/ablation_val.csv"
if os.path.exists(out_csv):
    rows = pd.read_csv(out_csv).to_dict("records")
else:
    rows = []
done = [r["model"] for r in rows]

for name, flags, seed in experiments:
    if name in done:
        print("skip", name)
        continue
    print("=== training", name)
    ckpt = f"{BASE}/models/abl_{name}.pt"
    model = run_abl(cfg, flags, seed, 3, ckpt)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    y, p = predict_hybrid(model, val_dl)
    res = evaluate(y, (p > 0.5).astype(int), p, name)
    rows.append(res)
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    del model
    gc.collect()
    torch.cuda.empty_cache()

ModuleNotFoundError: No module named 'optuna'

In [2]:
!pip install -q optuna

import os
print("BASE defined:", "BASE" in globals())
db_path = f"{BASE}/results/optuna.db"
print("optuna.db exists:", os.path.exists(db_path))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 12.6 MB/s eta 0:00:00
BASE defined: False


NameError: name 'BASE' is not defined

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/phishing-project"
print(os.listdir(BASE))
print(os.listdir(f"{BASE}/data"))
print(os.listdir(f"{BASE}/results"))

Mounted at /content/drive
['data', 'models', 'results']
['Phishing_Email.csv', 'train.csv', 'val.csv', 'test.csv']
['optuna.db', 'optuna_trials.csv', 'baselines_val.csv', 'ablation_val.csv']


In [4]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score
from sklearn.metrics import recall_score, f1_score
from sklearn.metrics import roc_auc_score, confusion_matrix

TEXT_COL = "Email Text"
LABEL_COL = "Email Type"

train = pd.read_csv(f"{BASE}/data/train.csv")
val = pd.read_csv(f"{BASE}/data/val.csv")
test = pd.read_csv(f"{BASE}/data/test.csv")

label_map = {"Safe Email": 0, "Phishing Email": 1}
for d in (train, val, test):
    if d[LABEL_COL].dtype == object:
        d[LABEL_COL] = d[LABEL_COL].map(label_map)
    d.dropna(subset=[TEXT_COL, LABEL_COL], inplace=True)
    d[LABEL_COL] = d[LABEL_COL].astype(int)
    d.reset_index(drop=True, inplace=True)

print(len(train), len(val), len(test))
print(train[LABEL_COL].value_counts())

def evaluate(y_true, y_pred, y_score=None, name="model"):
    res = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
    }
    if y_score is not None:
        res["roc_auc"] = float(roc_auc_score(y_true, y_score))
    print(res)
    print(confusion_matrix(y_true, y_pred))
    return res

11180 3726 3728
Email Type
0    6786
1    4394
Name: count, dtype: int64


In [5]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from transformers import get_linear_schedule_with_warmup

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256
BATCH = 16
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class EmailDS(Dataset):
    def __init__(self, df):
        self.texts = df[TEXT_COL].astype(str).tolist()
        self.labels = df[LABEL_COL].astype(int).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = tokenizer(
            self.texts[i],
            truncation=True,
            max_length=MAX_LEN,
            padding="max_length",
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item

train_dl = DataLoader(EmailDS(train), batch_size=BATCH, shuffle=True)
val_dl = DataLoader(EmailDS(val), batch_size=32)

def predict_hybrid(model, dl):
    model.eval()
    probs = []
    labels = []
    with torch.no_grad():
        for b in dl:
            b = {k: v.to(device) for k, v in b.items()}
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(b["input_ids"], b["attention_mask"])
            probs.extend(torch.sigmoid(logits.float()).cpu().numpy())
            labels.extend(b["labels"].cpu().numpy())
    return np.array(labels), np.array(probs)

device: cuda


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [6]:
import random, gc, os
import numpy as np
import torch.nn as nn
from transformers import AutoModel

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

class AblModel(nn.Module):
    def __init__(self, gru_units, dropout, use_attn=True,
                 use_cnn=True, use_gru=True, heads=8, filters=128):
        super().__init__()
        self.use_attn = use_attn
        self.use_cnn = use_cnn
        self.use_gru = use_gru
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        hid = self.bert.config.hidden_size
        self.attn = nn.MultiheadAttention(
            hid, heads, dropout=0.1, batch_first=True)
        self.norm = nn.LayerNorm(hid)
        self.conv = nn.Conv1d(hid, filters, kernel_size=3, padding=1)
        self.bn = nn.BatchNorm1d(filters)
        dim = filters if use_cnn else hid
        if use_gru:
            self.gru = nn.GRU(dim, gru_units, batch_first=True,
                              bidirectional=True)
            dim = 2 * gru_units
        self.drop = nn.Dropout(dropout)
        self.fc1 = nn.Linear(2 * dim, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        h = out.last_hidden_state
        if self.use_attn:
            pad = attention_mask == 0
            a = self.attn(h, h, h, key_padding_mask=pad, need_weights=False)
            h = self.norm(h + a[0])
        if self.use_cnn:
            c = self.conv(h.transpose(1, 2))
            h = torch.relu(self.bn(c)).transpose(1, 2)
        if self.use_gru:
            h, _ = self.gru(h)
        m = attention_mask.unsqueeze(-1).to(h.dtype)
        mean = (h * m).sum(1) / m.sum(1).clamp(min=1)
        mx = h.masked_fill(m == 0, -1e4).max(1).values
        feats = self.drop(torch.cat([mean, mx], dim=1))
        z = self.drop(torch.relu(self.fc1(feats)))
        return self.fc2(z).squeeze(-1)

In [7]:
from torch.optim import AdamW
from sklearn.metrics import f1_score

def run_abl(cfg, flags, seed, epochs, save_path):
    set_seed(seed)
    model = AblModel(cfg["gru_units"], cfg["dropout"], **flags).to(device)
    bert_p = list(model.bert.parameters())
    head_p = []
    for n, p in model.named_parameters():
        if not n.startswith("bert."):
            head_p.append(p)
    groups = [
        {"params": bert_p, "lr": cfg["bert_lr"]},
        {"params": head_p, "lr": cfg["head_lr"]},
    ]
    opt = AdamW(groups, weight_decay=0.01)
    total = len(train_dl) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    scaler = torch.cuda.amp.GradScaler()
    loss_fn = nn.BCEWithLogitsLoss()
    best = 0.0
    for epoch in range(epochs):
        model.train()
        for step, b in enumerate(train_dl):
            b = {k: v.to(device) for k, v in b.items()}
            opt.zero_grad()
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(b["input_ids"], b["attention_mask"])
            loss = loss_fn(logits.float(), b["labels"].float())
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            if step % 200 == 0:
                print(f"  epoch {epoch+1} step {step}/{len(train_dl)} loss {loss.item():.4f}")
        y, p = predict_hybrid(model, val_dl)
        f1 = f1_score(y, (p > 0.5).astype(int))
        print(f"  --- epoch {epoch+1} val F1: {f1:.4f}")
        if f1 > best:
            best = f1
            torch.save(model.state_dict(), save_path)
    return model

In [8]:
import optuna

db = f"sqlite:///{BASE}/results/optuna.db"
st = optuna.load_study(study_name="hybrid_tuning", storage=db)
bp = st.best_params
cfg = {
    "gru_units": bp["gru_units"],
    "dropout": bp["dropout"],
    "bert_lr": bp["bert_lr"],
    "head_lr": bp["head_lr"],
}
print("Using:", cfg)

experiments = [
    ("full_seed42", {}, 42),
    ("full_seed43", {}, 43),
    ("full_seed44", {}, 44),
    ("no_attention", {"use_attn": False}, 42),
    ("no_cnn", {"use_cnn": False}, 42),
    ("no_bigru", {"use_gru": False}, 42),
]

out_csv = f"{BASE}/results/ablation_val.csv"
if os.path.exists(out_csv):
    rows = pd.read_csv(out_csv).to_dict("records")
else:
    rows = []
done = [r["model"] for r in rows]

for name, flags, seed in experiments:
    if name in done:
        print("skip", name)
        continue
    print("=== training", name)
    ckpt = f"{BASE}/models/abl_{name}.pt"
    model = run_abl(cfg, flags, seed, 3, ckpt)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    y, p = predict_hybrid(model, val_dl)
    res = evaluate(y, (p > 0.5).astype(int), p, name)
    rows.append(res)
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    del model
    gc.collect()
    torch.cuda.empty_cache()

Using: {'gru_units': 256, 'dropout': 0.4100531293444458, 'bert_lr': 4.536089127921625e-05, 'head_lr': 0.002354770601447807}
skip full_seed42
skip full_seed43
skip full_seed44
skip no_attention
skip no_cnn
skip no_bigru


In [9]:
df = pd.read_csv(f"{BASE}/results/ablation_val.csv")
cols = ["accuracy", "precision", "recall", "f1", "roc_auc"]

print("All runs:")
print(df[["model"] + cols].round(4).to_string(index=False))

full = df[df["model"].str.startswith("full")]
print()
print("Full model, mean over", len(full), "seeds:")
print(full[cols].mean().round(4))
print()
print("Full model, std over", len(full), "seeds:")
print(full[cols].std().round(4))

print()
print("Ablation (seed 42):")
names = ["full_seed42", "no_attention", "no_cnn", "no_bigru"]
abl = df[df["model"].isin(names)]
print(abl[["model"] + cols].round(4).to_string(index=False))

All runs:
       model  accuracy  precision  recall     f1  roc_auc
 full_seed42    0.9780     0.9552  0.9904 0.9725   0.9981
 full_seed43    0.9772     0.9618  0.9809 0.9712   0.9983
 full_seed44    0.9764     0.9550  0.9863 0.9704   0.9978
no_attention    0.9772     0.9618  0.9809 0.9712   0.9979
      no_cnn    0.9753     0.9531  0.9856 0.9691   0.9979
    no_bigru    0.9791     0.9589  0.9891 0.9738   0.9977

Full model, mean over 3 seeds:
accuracy     0.9772
precision    0.9573
recall       0.9859
f1           0.9714
roc_auc      0.9980
dtype: float64

Full model, std over 3 seeds:
accuracy     0.0008
precision    0.0039
recall       0.0048
f1           0.0010
roc_auc      0.0003
dtype: float64

Ablation (seed 42):
       model  accuracy  precision  recall     f1  roc_auc
 full_seed42    0.9780     0.9552  0.9904 0.9725   0.9981
no_attention    0.9772     0.9618  0.9809 0.9712   0.9979
      no_cnn    0.9753     0.9531  0.9856 0.9691   0.9979
    no_bigru    0.9791     0.9589  0.9

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from transformers import AutoModelForSequenceClassification

test_dl = DataLoader(EmailDS(test), batch_size=32)
test_rows = []
y_test = test[LABEL_COL]

# 1. TF-IDF baselines
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2),
                        stop_words="english", sublinear_tf=True)
Xtr = tfidf.fit_transform(train[TEXT_COL].astype(str))
Xte = tfidf.transform(test[TEXT_COL].astype(str))

lr = LogisticRegression(max_iter=1000).fit(Xtr, train[LABEL_COL])
test_rows.append(evaluate(y_test, lr.predict(Xte),
                          lr.predict_proba(Xte)[:, 1], "TFIDF+LogReg"))

svm = LinearSVC().fit(Xtr, train[LABEL_COL])
test_rows.append(evaluate(y_test, svm.predict(Xte),
                          svm.decision_function(Xte), "TFIDF+SVM"))

# 2. DistilBERT only
dbm = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2).to(device)
dbm.load_state_dict(torch.load(f"{BASE}/models/distilbert_best.pt",
                               map_location=device))
dbm.eval()
probs = []
labels = []
with torch.no_grad():
    for b in test_dl:
        b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast("cuda", dtype=torch.float16):
            out = dbm(input_ids=b["input_ids"],
                      attention_mask=b["attention_mask"])
        pr = torch.softmax(out.logits.float(), dim=1)[:, 1]
        probs.extend(pr.cpu().numpy())
        labels.extend(b["labels"].cpu().numpy())
probs = np.array(probs)
labels = np.array(labels)
test_rows.append(evaluate(labels, (probs > 0.5).astype(int),
                          probs, "DistilBERT"))
del dbm
gc.collect()
torch.cuda.empty_cache()

# 3. Hybrid models (full x3 seeds, then ablations)
runs = [
    ("full_seed42", {}),
    ("full_seed43", {}),
    ("full_seed44", {}),
    ("no_attention", {"use_attn": False}),
    ("no_cnn", {"use_cnn": False}),
    ("no_bigru", {"use_gru": False}),
]
for name, flags in runs:
    m = AblModel(cfg["gru_units"], cfg["dropout"], **flags).to(device)
    m.load_state_dict(torch.load(f"{BASE}/models/abl_{name}.pt",
                                 map_location=device))
    y, p = predict_hybrid(m, test_dl)
    test_rows.append(evaluate(y, (p > 0.5).astype(int), p, name))
    del m
    gc.collect()
    torch.cuda.empty_cache()

# 4. Save and summarize
tdf = pd.DataFrame(test_rows)
tdf.to_csv(f"{BASE}/results/test_results.csv", index=False)
cols = ["accuracy", "precision", "recall", "f1", "roc_auc"]
print()
print(tdf[["model"] + cols].round(4).to_string(index=False))
full = tdf[tdf["model"].str.startswith("full")]
print()
print("Full model on test, mean:")
print(full[cols].mean().round(4))
print("Full model on test, std:")
print(full[cols].std().round(4))

{'model': 'TFIDF+LogReg', 'accuracy': 0.9753218884120172, 'precision': 0.9601620526671168, 'recall': 0.977319587628866, 'f1': 0.9686648501362398, 'roc_auc': 0.9962834288064126}
[[2214   59]
 [  33 1422]]
{'model': 'TFIDF+SVM', 'accuracy': 0.9785407725321889, 'precision': 0.9610999329309189, 'recall': 0.9848797250859107, 'f1': 0.9728445349626612, 'roc_auc': 0.9969144733559808}
[[2215   58]
 [  22 1433]]


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'model': 'no_cnn', 'accuracy': 0.9755901287553648, 'precision': 0.9546666666666667, 'recall': 0.9841924398625429, 'f1': 0.9692047377326565, 'roc_auc': 0.9972962144886257}
[[2205   68]
 [  23 1432]]


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'model': 'no_bigru', 'accuracy': 0.9788090128755365, 'precision': 0.961744966442953, 'recall': 0.9848797250859107, 'f1': 0.9731748726655348, 'roc_auc': 0.997392972637098}
[[2216   57]
 [  22 1433]]

       model  accuracy  precision  recall     f1  roc_auc
TFIDF+LogReg    0.9753     0.9602  0.9773 0.9687   0.9963
   TFIDF+SVM    0.9785     0.9611  0.9849 0.9728   0.9969
  DistilBERT    0.9767     0.9634  0.9773 0.9703   0.9979
 full_seed42    0.9764     0.9572  0.9835 0.9702   0.9978
 full_seed43    0.9764     0.9609  0.9794 0.9700   0.9972
 full_seed44    0.9767     0.9609  0.9801 0.9704   0.9978
no_attention    0.9769     0.9666  0.9746 0.9706   0.9981
      no_cnn    0.9756     0.9547  0.9842 0.9692   0.9973
    no_bigru    0.9788     0.9617  0.9849 0.9732   0.9974

Full model on test, mean:
accuracy     0.9765
precision    0.9597
recall       0.9810
f1           0.9702
roc_auc      0.9976
dtype: float64
Full model on test, std:
accuracy     0.0002
precision    0.0021
recall       

In [11]:
m = AblModel(cfg["gru_units"], cfg["dropout"]).to(device)
path = f"{BASE}/models/abl_full_seed42.pt"
m.load_state_dict(torch.load(path, map_location=device))
y, p = predict_hybrid(m, test_dl)

ea = test.copy()
ea["prob"] = p
ea["pred"] = (p > 0.5).astype(int)
ea["words"] = ea[TEXT_COL].astype(str).str.split().str.len()
ea["wrong"] = ea["pred"] != ea[LABEL_COL]

fp = ea[(ea[LABEL_COL] == 0) & (ea["pred"] == 1)]
fn = ea[(ea[LABEL_COL] == 1) & (ea["pred"] == 0)]
print("False positives (safe flagged as phishing):", len(fp))
print("False negatives (phishing missed):", len(fn))

print()
print("Average words, correct emails:", round(ea[~ea.wrong]["words"].mean()))
print("Average words, wrong emails:", round(ea[ea.wrong]["words"].mean()))
print("Median words, correct:", ea[~ea.wrong]["words"].median())
print("Median words, wrong:", ea[ea.wrong]["words"].median())

ea[ea.wrong].to_csv(f"{BASE}/results/errors_full_seed42.csv", index=False)

def show(df, title, n=6):
    print()
    print("=====", title)
    for _, r in df.head(n).iterrows():
        print(f"[prob={r['prob']:.2f}, words={r['words']}]")
        print(str(r[TEXT_COL])[:300].replace("\n", " "))
        print("---")

show(fp, "FALSE POSITIVES")
show(fn, "FALSE NEGATIVES")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


False positives (safe flagged as phishing): 64
False negatives (phishing missed): 24

Average words, correct emails: 361
Average words, wrong emails: 510
Median words, correct: 166.5
Median words, wrong: 35.0

===== FALSE POSITIVES
[prob=0.98, words=637]
renewal notice for your domain name ( s ) dear register . com subscriber , the domain name ( s ) listed below will expire on : wed , aug 08 , 2001 . register . com offers the following renewal features : 1 - quick renew , the easy renewal feature you can use at any time or 2 - saferenew ( tm ) , our
---
[prob=0.61, words=1]
empty
---
[prob=1.00, words=266]
us news archive @ ft . com reliable country intelligence for a challenging world with country reports supporting your decisions , you  ' re working with the best source of country intelligence available . turn to the economist intelligence unit at : http : / / store . eiu . com dear ft . com user f
---
[prob=0.61, words=1]
empty
---
[prob=0.61, words=1]
empty
---
[prob=1.00, words=9

In [12]:
def stats(name, d):
    t = d[TEXT_COL].astype(str).str.strip().str.lower()
    print(name, "| rows:", len(d), "| 'empty':", (t == "empty").sum(),
          "| duplicate texts:", t.duplicated().sum())

stats("train", train)
stats("val", val)
stats("test", test)

tr = set(train[TEXT_COL].astype(str).str.strip().str.lower())
te = test[TEXT_COL].astype(str).str.strip().str.lower()
print("test emails also found in train:", te.isin(tr).sum())

is_empty = train[TEXT_COL].astype(str).str.strip().str.lower() == "empty"
print("labels of 'empty' rows in train:")
print(train[is_empty][LABEL_COL].value_counts())

train | rows: 11180 | 'empty': 320 | duplicate texts: 546
val | rows: 3726 | 'empty': 116 | duplicate texts: 147
test | rows: 3728 | 'empty': 97 | duplicate texts: 119
test emails also found in train: 242
labels of 'empty' rows in train:
Email Type
1    198
0    122
Name: count, dtype: int64


In [13]:
def norm(s):
    return s.astype(str).str.strip().str.lower()

t_te = norm(test[TEXT_COL])
t_tr = set(norm(train[TEXT_COL]))
t_va = set(norm(val[TEXT_COL]))

keep = (
    (t_te != "empty")
    & (~t_te.isin(t_tr))
    & (~t_te.isin(t_va))
    & (~t_te.duplicated())
)
test_c = test[keep].reset_index(drop=True)

print("original test:", len(test), "-> clean test:", len(test_c))
print(test_c[LABEL_COL].value_counts())
test_c.to_csv(f"{BASE}/data/test_clean.csv", index=False)

original test: 3728 -> clean test: 3444
Email Type
0    2198
1    1246
Name: count, dtype: int64


In [14]:
from transformers.utils import logging
logging.set_verbosity_error()

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from transformers import AutoModelForSequenceClassification

test_dl_c = DataLoader(EmailDS(test_c), batch_size=32)
y_c = test_c[LABEL_COL]
rows_c = []

tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2),
                        stop_words="english", sublinear_tf=True)
Xtr = tfidf.fit_transform(train[TEXT_COL].astype(str))
Xte = tfidf.transform(test_c[TEXT_COL].astype(str))

lr = LogisticRegression(max_iter=1000).fit(Xtr, train[LABEL_COL])
rows_c.append(evaluate(y_c, lr.predict(Xte),
                       lr.predict_proba(Xte)[:, 1], "TFIDF+LogReg"))
svm = LinearSVC().fit(Xtr, train[LABEL_COL])
rows_c.append(evaluate(y_c, svm.predict(Xte),
                       svm.decision_function(Xte), "TFIDF+SVM"))

dbm = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2).to(device)
dbm.load_state_dict(torch.load(f"{BASE}/models/distilbert_best.pt",
                               map_location=device))
dbm.eval()
probs = []
labels = []
with torch.no_grad():
    for b in test_dl_c:
        b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast("cuda", dtype=torch.float16):
            out = dbm(input_ids=b["input_ids"],
                      attention_mask=b["attention_mask"])
        pr = torch.softmax(out.logits.float(), dim=1)[:, 1]
        probs.extend(pr.cpu().numpy())
        labels.extend(b["labels"].cpu().numpy())
probs = np.array(probs)
labels = np.array(labels)
rows_c.append(evaluate(labels, (probs > 0.5).astype(int),
                       probs, "DistilBERT"))
del dbm
gc.collect()
torch.cuda.empty_cache()

runs = [
    ("full_seed42", {}),
    ("full_seed43", {}),
    ("full_seed44", {}),
    ("no_attention", {"use_attn": False}),
    ("no_cnn", {"use_cnn": False}),
    ("no_bigru", {"use_gru": False}),
]
for name, flags in runs:
    m = AblModel(cfg["gru_units"], cfg["dropout"], **flags).to(device)
    m.load_state_dict(torch.load(f"{BASE}/models/abl_{name}.pt",
                                 map_location=device))
    y, p = predict_hybrid(m, test_dl_c)
    rows_c.append(evaluate(y, (p > 0.5).astype(int), p, name))
    if name == "full_seed42":
        p42 = p
    del m
    gc.collect()
    torch.cuda.empty_cache()

cdf = pd.DataFrame(rows_c)
cdf.to_csv(f"{BASE}/results/test_clean_results.csv", index=False)
cols = ["accuracy", "precision", "recall", "f1", "roc_auc"]
print()
print(cdf[["model"] + cols].round(4).to_string(index=False))
full = cdf[cdf["model"].str.startswith("full")]
print()
print("Full model on clean test, mean:")
print(full[cols].mean().round(4))
print("Full model on clean test, std:")
print(full[cols].std().round(4))

ec = test_c.copy()
ec["pred"] = (p42 > 0.5).astype(int)
ec["words"] = ec[TEXT_COL].astype(str).str.split().str.len()
ec["wrong"] = ec["pred"] != ec[LABEL_COL]
print()
print("Errors (full_seed42):", int(ec["wrong"].sum()), "of", len(ec))
print("Median words, correct:", ec[~ec.wrong]["words"].median())
print("Median words, wrong:", ec[ec.wrong]["words"].median())

{'model': 'TFIDF+LogReg', 'accuracy': 0.9831591173054588, 'precision': 0.9790322580645161, 'recall': 0.9743178170144462, 'f1': 0.9766693483507642, 'roc_auc': 0.9970946154171967}
[[2172   26]
 [  32 1214]]
{'model': 'TFIDF+SVM', 'accuracy': 0.9863530778164924, 'precision': 0.9799839871897518, 'recall': 0.9823434991974318, 'f1': 0.9811623246492986, 'roc_auc': 0.9975546863703614}
[[2173   25]
 [  22 1224]]


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

{'model': 'DistilBERT', 'accuracy': 0.9843205574912892, 'precision': 0.9829821717990276, 'recall': 0.9735152487961477, 'f1': 0.978225806451613, 'roc_auc': 0.9987295104114786}
[[2177   21]
 [  33 1213]]


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

{'model': 'full_seed42', 'accuracy': 0.9846109175377468, 'precision': 0.9752988047808765, 'recall': 0.9823434991974318, 'f1': 0.9788084766093562, 'roc_auc': 0.9988202466272417}
[[2167   31]
 [  22 1224]]


...

{'model': 'no_attention', 'accuracy': 0.9849012775842044, 'precision': 0.9869494290375204, 'recall': 0.971107544141252, 'f1': 0.9789644012944984, 'roc_auc': 0.9989756118578541}
[[2182   16]
 [  36 1210]]


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

{'model': 'no_cnn', 'accuracy': 0.9834494773519163, 'precision': 0.9722001588562351, 'recall': 0.9823434991974318, 'f1': 0.9772455089820359, 'roc_auc': 0.9982667009407356}
[[2163   35]
 [  22 1224]]


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

{'model': 'no_bigru', 'accuracy': 0.9866434378629501, 'precision': 0.9807692307692307, 'recall': 0.9823434991974318, 'f1': 0.9815557337610264, 'roc_auc': 0.9981699399863001}
[[2174   24]
 [  22 1224]]

       model  accuracy  precision  recall     f1  roc_auc
TFIDF+LogReg    0.9832     0.9790  0.9743 0.9767   0.9971
   TFIDF+SVM    0.9864     0.9800  0.9823 0.9812   0.9976
  DistilBERT    0.9843     0.9830  0.9735 0.9782   0.9987
 full_seed42    0.9846     0.9753  0.9823 0.9788   0.9988
 full_seed43    0.9843     0.9799  0.9767 0.9783   0.9981
 full_seed44    0.9846     0.9799  0.9775 0.9787   0.9987
no_attention    0.9849     0.9869  0.9711 0.9790   0.9990
      no_cnn    0.9834     0.9722  0.9823 0.9772   0.9983
    no_bigru    0.9866     0.9808  0.9823 0.9816   0.9982

Full model on clean test, mean:
accuracy     0.9845
precision    0.9784
recall       0.9789
f1           0.9786
roc_auc      0.9985
dtype: float64
Full model on clean test, std:
accuracy     0.0002
precision    0.0026

In [15]:
fp = ec[(ec[LABEL_COL] == 0) & (ec["pred"] == 1)]
fn = ec[(ec[LABEL_COL] == 1) & (ec["pred"] == 0)]
print("False positives:", len(fp), "| False negatives:", len(fn))

ec[ec.wrong].to_csv(f"{BASE}/results/errors_clean_seed42.csv", index=False)

def show(df, title, n=8):
    print()
    print("=====", title)
    for _, r in df.head(n).iterrows():
        print(f"[words={r['words']}]")
        print(str(r[TEXT_COL])[:250].replace("\n", " "))
        print("---")

show(fp, "FALSE POSITIVES")
show(fn, "FALSE NEGATIVES")

False positives: 31 | False negatives: 22

===== FALSE POSITIVES
[words=637]
renewal notice for your domain name ( s ) dear register . com subscriber , the domain name ( s ) listed below will expire on : wed , aug 08 , 2001 . register . com offers the following renewal features : 1 - quick renew , the easy renewal feature you
---
[words=266]
us news archive @ ft . com reliable country intelligence for a challenging world with country reports supporting your decisions , you  ' re working with the best source of country intelligence available . turn to the economist intelligence unit at :
---
[words=9]
can you work you legalies into this form .
---
[words=728]
Tech Update Today VITAL SIGNS FOR JULY 10, 2002David Berlind What to look for in your next smart phone With PDAs and cell phones on the fast path to convergence, choosing the right  device is harder than ever. Before you buy one more handheld, see my
---
[words=208]
your amazon . com order ( # 104 - 9670681 - 0325567 ) hello from 

In [16]:
!pip install -q lime

from lime.lime_text import LimeTextExplainer

best_model = AblModel(cfg["gru_units"], cfg["dropout"]).to(device)
path = f"{BASE}/models/abl_full_seed42.pt"
best_model.load_state_dict(torch.load(path, map_location=device))
best_model.eval()

def predict_proba_texts(texts):
    out = []
    for i in range(0, len(texts), 16):
        chunk = [str(t) for t in texts[i:i + 16]]
        enc = tokenizer(chunk, truncation=True, max_length=MAX_LEN,
                        padding="max_length", return_tensors="pt")
        ids = enc["input_ids"].to(device)
        mask = enc["attention_mask"].to(device)
        with torch.no_grad():
            with torch.autocast("cuda", dtype=torch.float16):
                logits = best_model(ids, mask)
        p = torch.sigmoid(logits.float()).cpu().numpy()
        out.append(np.stack([1 - p, p], axis=1))
    return np.vstack(out)

explainer = LimeTextExplainer(class_names=["Safe", "Phishing"])

for label_val in [1, 0]:
    row = test_c[test_c[LABEL_COL] == label_val].iloc[3]
    text = str(row[TEXT_COL])
    prob = float(predict_proba_texts([text])[0, 1])
    exp = explainer.explain_instance(
        text, predict_proba_texts, num_features=10, num_samples=400)
    print("TRUE LABEL:", label_val, "| P(phishing) =", round(prob, 3))
    print(text[:200].replace("\n", " "))
    print(exp.as_list())
    print()
    exp.save_to_file(f"{BASE}/results/lime_label{label_val}.html")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 8.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

TRUE LABEL: 1 | P(phishing) = 1.0
important message jaguar are anaerobic is brute ofclose stroboscopic your bookstore is aliphatic and libation depressant in chlordane is cheryl of roadster benefit your petrology is prolong are ashevi
[(np.str_('know'), 0.00038625633392553964), (np.str_('indigestible'), 0.00036935375483313605), (np.str_('the'), -0.0002319252395611474), (np.str_('program'), -0.0002158133868194619), (np.str_('today'), -8.309094112595975e-05), (np.str_('salary'), -7.65613202859111e-05), (np.str_('but'), -4.88206060708732e-05), (np.str_('representatives'), -1.8834125695924687e-05), (np.str_('abilities'), -1.3755069755622314e-05), (np.str_('areaspirate'), -3.2737408298368574e-06)]

TRUE LABEL: 0 | P(phishing) = 0.981
renewal notice for your domain name ( s ) dear register . com subscriber , the domain name ( s ) listed below will expire on : wed , aug 08 , 2001 . register . com offers the following renewal feature
[(np.str_('your'), 0.39910788955367943), (np.str_('2001'), -

In [17]:
!pip install -q gradio

import gradio as gr

def analyze(text):
    text = " ".join(str(text).split()[:300])
    if not text:
        return {}, []
    p = float(predict_proba_texts([text])[0, 1])
    exp = explainer.explain_instance(
        text, predict_proba_texts, num_features=12, num_samples=300)
    w = {}
    for word, val in exp.as_list():
        w[str(word).lower()] = float(val)
    out = []
    for tok in text.split():
        key = tok.strip(".,!?;:()[]\"'").lower()
        if key in w and abs(w[key]) > 0.01:
            tag = "Phishing cue" if w[key] > 0 else "Safe cue"
            out.append((tok + " ", tag))
        else:
            out.append((tok + " ", None))
    return {"Phishing": p, "Safe": 1 - p}, out

colors = {"Phishing cue": "#ff6b6b", "Safe cue": "#51cf66"}
demo = gr.Interface(
    fn=analyze,
    inputs=gr.Textbox(lines=10, label="Paste email text"),
    outputs=[
        gr.Label(label="Prediction"),
        gr.HighlightedText(label="Words that influenced the decision",
                           color_map=colors),
    ],
    title="Phishing Email Detector",
    description="BERT + attention + CNN + BiGRU hybrid model with LIME explanations.",
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2e796503e57ed4bb04.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [1]:
!pip install -q optuna lime gradio

from google.colab import drive
drive.mount('/content/drive')

import os, time
import numpy as np
import torch
import torch.nn as nn
import optuna
import gradio as gr
from transformers import AutoTokenizer, AutoModel
from transformers.utils import logging
from lime.lime_text import LimeTextExplainer

logging.set_verbosity_error()

BASE = "/content/drive/MyDrive/phishing-project"
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class AblModel(nn.Module):
    def __init__(self, gru_units, dropout, use_attn=True,
                 use_cnn=True, use_gru=True, heads=8, filters=128):
        super().__init__()
        self.use_attn = use_attn
        self.use_cnn = use_cnn
        self.use_gru = use_gru
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        hid = self.bert.config.hidden_size
        self.attn = nn.MultiheadAttention(
            hid, heads, dropout=0.1, batch_first=True)
        self.norm = nn.LayerNorm(hid)
        self.conv = nn.Conv1d(hid, filters, kernel_size=3, padding=1)
        self.bn = nn.BatchNorm1d(filters)
        dim = filters if use_cnn else hid
        if use_gru:
            self.gru = nn.GRU(dim, gru_units, batch_first=True,
                              bidirectional=True)
            dim = 2 * gru_units
        self.drop = nn.Dropout(dropout)
        self.fc1 = nn.Linear(2 * dim, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        h = out.last_hidden_state
        if self.use_attn:
            pad = attention_mask == 0
            a = self.attn(h, h, h, key_padding_mask=pad, need_weights=False)
            h = self.norm(h + a[0])
        if self.use_cnn:
            c = self.conv(h.transpose(1, 2))
            h = torch.relu(self.bn(c)).transpose(1, 2)
        if self.use_gru:
            h, _ = self.gru(h)
        m = attention_mask.unsqueeze(-1).to(h.dtype)
        mean = (h * m).sum(1) / m.sum(1).clamp(min=1)
        mx = h.masked_fill(m == 0, -1e4).max(1).values
        feats = self.drop(torch.cat([mean, mx], dim=1))
        z = self.drop(torch.relu(self.fc1(feats)))
        return self.fc2(z).squeeze(-1)

db = f"sqlite:///{BASE}/results/optuna.db"
st = optuna.load_study(study_name="hybrid_tuning", storage=db)
bp = st.best_params
cfg = {
    "gru_units": bp["gru_units"],
    "dropout": bp["dropout"],
}
print("Using:", cfg)

best_model = AblModel(cfg["gru_units"], cfg["dropout"]).to(device)
path = f"{BASE}/models/abl_full_seed42.pt"
best_model.load_state_dict(torch.load(path, map_location=device))
best_model.eval()

def predict_proba_texts(texts):
    out = []
    for i in range(0, len(texts), 16):
        chunk = [str(t) for t in texts[i:i + 16]]
        enc = tokenizer(chunk, truncation=True, max_length=MAX_LEN,
                        padding="max_length", return_tensors="pt")
        ids = enc["input_ids"].to(device)
        mask = enc["attention_mask"].to(device)
        with torch.no_grad():
            with torch.autocast("cuda", dtype=torch.float16):
                logits = best_model(ids, mask)
        p = torch.sigmoid(logits.float()).cpu().numpy()
        out.append(np.stack([1 - p, p], axis=1))
    return np.vstack(out)

explainer = LimeTextExplainer(class_names=["Safe", "Phishing"])

def analyze(text):
    text = " ".join(str(text).split()[:200])
    if not text:
        return {}, []
    p = float(predict_proba_texts([text])[0, 1])
    exp = explainer.explain_instance(
        text, predict_proba_texts, num_features=10, num_samples=150)
    w = {}
    for word, val in exp.as_list():
        w[str(word).lower()] = float(val)
    out = []
    for tok in text.split():
        key = tok.strip(".,!?;:()[]\"'").lower()
        if key in w and abs(w[key]) > 0.01:
            tag = "Phishing cue" if w[key] > 0 else "Safe cue"
            out.append((tok + " ", tag))
        else:
            out.append((tok + " ", None))
    return {"Phishing": p, "Safe": 1 - p}, out

t0 = time.time()
sample = ("Dear customer, your account will be suspended. "
          "Click here immediately to verify your password.")
res = analyze(sample)
print("TEST OK in", round(time.time() - t0, 1), "seconds")
print(res[0])

try:
    gr.close_all()
except Exception as e:
    print("close:", e)

colors = {"Phishing cue": "#ff6b6b", "Safe cue": "#51cf66"}
demo = gr.Interface(
    fn=analyze,
    inputs=gr.Textbox(lines=10, label="Paste email text"),
    outputs=[
        gr.Label(label="Prediction"),
        gr.HighlightedText(label="Words that influenced the decision",
                           color_map=colors),
    ],
    title="Phishing Email Detector",
    description="BERT + attention + CNN + BiGRU hybrid model with LIME explanations.",
)
demo.launch(share=True, debug=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 9.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 19.2 MB/s eta 0:00:00
Mounted at /content/drive
device: cuda


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

TEST OK in 1.9 seconds
{'Phishing': 0.9998736381530762, 'Safe': 0.00012636184692382812}
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://761b22b7c5ac49f3df.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://761b22b7c5ac49f3df.gradio.live
